## Softmax 从0开始实现


In [29]:
import torch
from IPython import display
from d2l import torch as d2l

batch_size = 256
train_iter,test_iter = d2l.load_data_fashion_mnist(batch_size)

### 初始化模型参数
 原始数据集中的每个样本都是28x28的图像。 本节将展平每个图像，把它们看作长度为784的向量,在softmax回归中，我们的输出与类别一样多。 因为我们的数据集有10个类别，所以网络输出维度为10。 因此，权重将构成一个784x10的矩阵，偏置将构成一个
1x10的行向量。 与线性回归一样，我们将使用正态分布初始化我们的权重W，偏置初始化为0。

In [30]:
num_input = 784
num_output = 10

W = torch.normal(0,0.01,size=(num_input,num_output),requires_grad=True)
b = torch.zeros(num_output,requires_grad=True)

### 定义Softmax操作
实现softmax由三个步骤组成：
1. 对每个项求幂（使用exp）；
2. 对每一行求和（小批量中每个样本是一行），得到每个样本的规范化常数；
3. 将每一行除以其规范化常数，确保结果的和为1。

在查看代码之前，我们回顾一下这个表达式：$$\mathrm{softmax}(\mathbf{X})_{ij}=\frac{\exp(\mathbf{X}_{ij})}{\sum_k\exp(\mathbf{X}_{ik})}.$$
下面的代码对于任何随机输入，我们将每个元素变成一个非负数。 此外，依据概率原理，每行总和为1。

In [31]:
def softmax(X):
    X_exp = torch.exp(X)
    partition = X_exp.sum(1,keepdim=True)
    return X_exp / partition # 应用广播机制

X = torch.normal(0,1,(2,5)) # 0-1之间 2行5列
X_prob = softmax(X)
X_prob,X_prob.sum(1)



(tensor([[0.3848, 0.1039, 0.1141, 0.1720, 0.2252],
         [0.0578, 0.0736, 0.4918, 0.1433, 0.2334]]),
 tensor([1.0000, 1.0000]))

### 定义模型
定义softmax操作后，我们可以实现softmax回归模型。 下面的代码定义了输入如何通过网络映射到输出。 注意，将数据传递到模型之前，我们使用reshape函数将每张原始图像展平为向量。

In [32]:
def net(X):
    return softmax(torch.matmul(X.reshape((-1,W.shape[0])),W)+b) 
# 上面这一句将输入变量 X 进行 reshape，-1 表示自动计算这一维度的大小以保持其他维度乘积不变，W.shape[0] 表示权重矩阵 W 的第一维大小
#（通常表示的是权重矩阵的输出通道数或隐藏单元数）。这样操作的结果是将输入 X 转换为可以与权重矩阵 W 进行矩阵乘法的形状。

### 定义损失函数
交叉熵损失函数这可能是深度学习中最常见的损失函数，因为目前分类问题的数量远远超过回归问题的数量。交叉熵采用真实标签的预测概率的负对数似然。 这里我们不使用Python的for循环迭代预测（这往往是低效的）， 而是通过一个运算符选择所有元素。 下面，我们创建一个数据样本y_hat，其中包含2个样本在3个类别的预测概率， 以及它们对应的标签y。 有了y，我们知道在第一个样本中，第一类是正确的预测； 而在第二个样本中，第三类是正确的预测。 然后使用y作为y_hat中概率的索引， 我们选择第一个样本中第一个类的概率和第二个样本中第三个类的概率。

In [33]:
y = torch.tensor([0,2]) # 目标类型为 0 和 2
y_hat = torch.tensor([[0.1,0.3,0.6],[0.3,0.1,0.5]])
y_hat[[0,1],y]

tensor([0.1000, 0.5000])

In [34]:
# 交叉熵损失函数
def cross_entropy(y_hat,y):
    return - torch.log(y_hat[range(len(y_hat)),y])
cross_entropy(y_hat=y_hat,y=y)

tensor([2.3026, 0.6931])

### 分类精度
给定预测概率分布y_hat，当我们必须输出硬预测（hard prediction）时， 我们通常选择预测概率最高的类。当预测与标签分类y一致时，即是正确的。 分类精度即正确预测数量与总预测数量之比。 虽然直接优化精度可能很困难（因为精度的计算不可导）， 但精度通常是我们最关心的性能衡量标准，我们在训练分类器时几乎总会关注它。

为了计算精度，我们执行以下操作。 首先，如果y_hat是矩阵，那么假定第二个维度存储每个类的预测分数。 我们使用argmax获得每行中最大元素的索引来获得预测类别。 然后我们将预测类别与真实y元素进行比较。 由于等式运算符“==”对数据类型很敏感， 因此我们将y_hat的数据类型转换为与y的数据类型一致。 结果是一个包含0（错）和1（对）的张量。 最后，我们求和会得到正确预测的数量。

In [35]:
def accuracy(y_hat,y):
    """计算预测正确的数量"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis =1)
    cmp = y_hat.type(y.dtype) == y # 将cmp转成bool类型
    return float(cmp.type(y.dtype).sum()) # 再将cmp转成y同类型的求和

accuracy(y_hat=y_hat,y=y) / len(y)

0.5

In [36]:
def evaluate_accuracy(net,data_iter):
    """计算在指定数据集上模型的精度"""
    if isinstance(net,torch.nn.Module):
        net.eval() # 将模型设置为评估模式
    metric = Accumulator(2) # 正确预测数，预测总数
    with torch.no_grad():
        for X,y in data_iter:
            metric.add(accuracy(net(X),y),y.numel()) #调用 accuracy 函数计算这一批次中模型预测正确的样本数，并将其累加到 metric 的第一个元素（正确预测数）
# 批次的真实标签 y 的元素个数（即样本数）累加到 metric 的第二个元素（预测总数）。
    return metric[0] / metric[1]

这里定义一个实用程序类Accumulator，用于对多个变量进行累加。 在上面的evaluate_accuracy函数中， 我们在Accumulator实例中创建了2个变量， 分别用于存储正确预测的数量和预测的总数量。 当我们遍历数据集时，两者都将随着时间的推移而累加。

In [38]:
class Accumulator:
    """在n个变量上累加"""
    def __init__(self,n) -> None:
        self.data = [0.0] * n
        pass

    def add(self,*args):
        self.data = [a + float(b) for a,b in zip(self.data,args)]

    def reset(self):
        self.data = [0.0] * len(self.data)
    
    def __getitem__(self,idx):
        return self.data[idx]

evaluate_accuracy(net,test_iter)

0.1059